# Microsoft Agent Framework — Beginner Learning Path

This notebook is a guided introduction to **Microsoft Agent Framework (MAF)**. We will move from one agent to tools, sessions, workflows, Human-in-the-Loop (HITL), and multi-agent orchestration.

**Learning style:** plain language, a real-life analogy, one concept at a time, runnable examples, and key takeaways.

> API baseline: MAF Python 1.19.0. Model-backed cells use Microsoft Foundry; workflow-only cells need no API key.

## The big picture

Think of an agent system as a small workplace:

| MAF concept | Real-life analogy |
|---|---|
| Agent | A teammate with a role and instructions |
| Model client | The teammate's reasoning engine |
| Tool | An approved application the teammate can use |
| Session | The conversation notebook |
| Workflow | The operating procedure |
| Orchestration | How multiple teammates coordinate |
| HITL | A manager review or missing-information request |

**Rule of thumb:** use agents for reasoning and language; use workflows for predictable control, durability, and auditability.

## 0. Setup

1. Use Python 3.10 or newer.
2. Install `requirements.txt`.
3. For model-backed lessons, install `requirements-models.txt`.
4. Run `az login`.
5. Copy `.env.example` to `.env` and set your Foundry endpoint and model.

The next cell is included for notebook users. Uncomment only what you need.

In [ ]:
# %pip install -q -r ../requirements.txt
# %pip install -q -r ../requirements-models.txt

## 1. Your first agent

An `Agent` combines:

- a model client,
- a name,
- instructions, and
- optionally tools and other behaviors.

This cell requires Microsoft Foundry credentials.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

guide = Agent(
    client=client,
    name="LearningGuide",
    instructions="Teach beginners using short explanations and one concrete example.",
)

# Uncomment after configuring Microsoft Foundry:
# answer = await guide.run("What is an AI agent?")
# print(answer.text)

### Key point

The model client talks to the model provider. The `Agent` adds the application role, instructions, tools, sessions, middleware, and framework behavior around it.

## 2. Give an agent a tool

A tool is a typed Python function the model may request. Good tool names, docstrings, and parameter descriptions help the model choose correctly.

This read-only demo tool uses `approval_mode="never_require"`. Side-effecting tools should normally use `"always_require"`.

In [ ]:
from typing import Annotated
from agent_framework import tool

@tool(approval_mode="never_require")
def course_progress(
    completed_modules: Annotated[int, "Number of completed modules"],
) -> str:
    """Return progress for this six-module course."""
    completed = max(0, min(completed_modules, 6))
    return f"{completed}/6 complete ({completed / 6:.0%})."

coach = Agent(
    client=client,
    name="StudyCoach",
    instructions="Use course_progress when the learner mentions completed modules.",
    tools=[course_progress],
)

# result = await coach.run("I completed 2 modules. How am I doing?")
# print(result.text)

## 3. Keep context with a session

Separate `agent.run(...)` calls do not automatically mean one conversation. Create a session and reuse it when later turns should remember earlier turns.

In [ ]:
session = coach.create_session()

# first = await coach.run(
#     "My name is Priya and I completed 2 modules.",
#     session=session,
# )
# second = await coach.run(
#     "What is my name and how far have I progressed?",
#     session=session,
# )
# print(second.text)

### Session safety

Use a different session for each user or conversation. A session is conversation state—not long-term business memory. Persistent memory is usually supplied through context providers or an application data store.

## 4. Build a deterministic workflow — no model required

A workflow owns the order of operations. A functional workflow looks like normal async Python. `@step` saves a completed result so it can be reused during replay or resume.

In [ ]:
from agent_framework import step, workflow

@step
async def collect_facts(topic: str) -> list[str]:
    print("collect_facts ran")
    return [
        f"{topic} combines model reasoning with application code.",
        "Workflows make control flow explicit.",
    ]

@step
async def write_summary(facts: list[str]) -> str:
    return " ".join(facts)

@workflow
async def learning_pipeline(topic: str) -> str:
    facts = await collect_facts(topic)
    return await write_summary(facts)

pipeline = learning_pipeline.build()
pipeline_result = await pipeline.run("Microsoft Agent Framework")
print(pipeline_result.get_outputs()[0])

## 5. Human in the Loop

HITL is one MAF capability inside the broader workflow system.

Two common patterns:

1. **Free-form request:** ask a person for missing data, feedback, or expert judgment with `request_info()`.
2. **Tool approval:** pause before an agent executes a specific function marked `approval_mode="always_require"`.

The focused notebook `01_maf_human_in_the_loop.ipynb` runs the full pause → inspect request → respond → resume cycle without a model.

In [ ]:
from agent_framework import RunContext, WorkflowRunState

@workflow
async def review_flow(draft: str, ctx: RunContext) -> str:
    feedback = await ctx.request_info(
        {"draft": draft, "question": "What should change?"},
        response_type=str,
        request_id="editor_review",
    )
    return f"{draft}\nReviewer feedback: {feedback}"

review = review_flow.build()
paused = await review.run("Draft: Agents can call tools.")
assert paused.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS

request = paused.get_request_info_events()[0]
print("Paused request:", request.data)

finished = await review.run(
    responses={request.request_id: "Explain that tools are application-defined."}
)
print(finished.get_outputs()[0])

### Approval is a security boundary

A useful approval screen shows the exact tool name, arguments, target, and impact. Authenticate the reviewer, expire approvals, record the decision, revalidate inputs, and make the final side effect idempotent.

## 6. Coordinate multiple agents

MAF includes sequential, concurrent, handoff, group-chat, and Magentic orchestration patterns.

A sequential pipeline is easiest to learn: the first agent writes, then the second agent reviews. This cell requires the configured model client.

In [ ]:
from agent_framework import AgentResponse
from agent_framework.orchestrations import SequentialBuilder

writer = Agent(
    client=client,
    name="Writer",
    instructions="Write one beginner-friendly paragraph.",
)
reviewer = Agent(
    client=client,
    name="Reviewer",
    instructions="Improve the previous agent's paragraph for clarity.",
)

sequential = SequentialBuilder(
    participants=[writer, reviewer],
    chain_only_agent_responses=True,
).build()

# events = await sequential.run("Explain why tools need clear descriptions.")
# final: AgentResponse = events.get_outputs()[0]
# print(final.text)

## Which orchestration should I choose?

| Pattern | Best fit |
|---|---|
| Sequential | Fixed pipeline: research → write → review |
| Concurrent | Independent work that can run in parallel |
| Handoff | A specialist takes ownership based on the conversation |
| Group chat | Several agents collaborate in shared discussion |
| Magentic | A manager dynamically plans and coordinates complex work |

Start with one agent. Add multiple agents only when distinct roles and coordination materially improve the solution.

## Production checklist

- Use narrow, typed tools and least-privilege credentials.
- Require approval for meaningful side effects.
- Keep user sessions isolated.
- Add retries only around idempotent operations.
- Persist checkpoints for long-running workflows.
- Add tracing, metrics, and structured audit logs.
- Test agent behavior and deterministic workflow branches separately.
- Pin package versions and review MAF release notes before upgrading.

## Exercises

1. Change the first agent's role and compare its answers.
2. Add a second read-only tool to the study coach.
3. Create two sessions and prove their histories remain separate.
4. Add a branch to the model-free workflow.
5. Change the HITL response to a Pydantic model with `approved` and `comments`.
6. Add a third agent to the sequential pipeline.
7. Sketch a production approval record: request id, reviewer, timestamp, arguments, and decision.

## Official references

- [Get started with Agent Framework](https://learn.microsoft.com/en-us/agent-framework/get-started/)
- [Official Python progressive samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/01-get-started)
- [Workflow capabilities](https://learn.microsoft.com/en-us/agent-framework/workflows/)
- [Workflow orchestrations](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/)
- [Human-in-the-loop workflows](https://learn.microsoft.com/en-us/agent-framework/workflows/human-in-the-loop)
- [Tool approval](https://learn.microsoft.com/en-us/agent-framework/agents/tools/tool-approval)